Gathering Data

In [14]:
import pandas as pd

# 1. LOAD DATASET
filename = 'data_v8.csv'
df = pd.read_csv(filename)

print("--- INFO DATASET AWAL ---")
print(f"Total baris: {len(df):,}")
print(f"Total duplicates: {df.duplicated().sum():,}")
print("\nTotal Missing Values per Kolom:")
print(df.isnull().sum())

--- INFO DATASET AWAL ---
Total baris: 134,565
Total duplicates: 13,281

Total Missing Values per Kolom:
user_id              0
date                 0
amount            4520
type                 0
category             0
subcategory       4517
payment_method    4399
description       4525
dtype: int64


### 1. Handling Noise (Duplicates & Missing Values)
menghapus semua baris duplikat dan baris yang memiliki nilai kosong.

In [15]:
# 2. CLEANING NOISE

# A. Drop Duplicates
df_clean = df.drop_duplicates()

# B. Drop Missing Values
df_clean = df_clean.dropna()

print("--- SETELAH CLEANING NOISE ---")
print(f"Total baris tersisa: {len(df_clean):,}")
print(f"Sisa missing values: {df_clean.isnull().sum().sum()}")
print(f"Sisa duplicates: {df_clean.duplicated().sum()}")

--- SETELAH CLEANING NOISE ---
Total baris tersisa: 111,480
Sisa missing values: 0
Sisa duplicates: 0


### 2. Formatting & Standardization
Memastikan tipe data sudah benar:
- Kolom `date` menjadi `datetime`.
- Kolom `amount` menjadi `integer`.
- Kolom berbasis teks diseragamkan format hurufnya dan dihilangkan spasi berlebihnya (*trailing/leading spaces*).

In [16]:
# 3. FORMATTING TIPE DATA & TEKS

# A. Tipe Data
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['amount'] = df_clean['amount'].astype(int)

# B. Standardisasi Teks (str.lower, str.title, str.strip)
text_columns = ['type', 'category', 'subcategory', 'payment_method']

for col in text_columns:
    if col == 'category':
        # Kategori pakai Title Case (Misal: Fast Food)
        df_clean[col] = df_clean[col].astype(str).str.title().str.strip()
    else:
        # Sisanya huruf kecil semua (lowercase)
        df_clean[col] = df_clean[col].astype(str).str.lower().str.strip()

print("--- TIPE DATA SETELAH DIPERBAIKI ---")
print(df_clean.dtypes)

--- TIPE DATA SETELAH DIPERBAIKI ---
user_id                   object
date              datetime64[ns]
amount                     int64
type                      object
category                  object
subcategory               object
payment_method            object
description               object
dtype: object


### 3. Final Check & Export
Mengurutkan kembali data berdasarkan `user_id` dan `date` agar terstruktur rapi, lalu mengekspornya sebagai file CSV yang siap dianalisis.

In [17]:
# 4. SORTING & EXPORT

# Mengurutkan ulang karena sebelumnya data sempat diacak saat inject noise
df_clean = df_clean.sort_values(by=['user_id', 'date']).reset_index(drop=True)

# Mengecek apakah ada nilai amount yang aneh (<= 0)
invalid_amounts = df_clean[df_clean['amount'] <= 0]
if not invalid_amounts.empty:
    print(f"Peringatan: Ditemukan {len(invalid_amounts)} baris dengan amount <= 0. Baris ini akan didrop.")
    df_clean = df_clean[df_clean['amount'] > 0]

# Export ke CSV baru
cleaned_filename = 'cleaned_data_v8.csv'
df_clean.to_csv(cleaned_filename, index=False)

print(f"\n✅ Proses Wrangling Selesai!")
print(f"Data bersih berhasil disimpan ke: '{cleaned_filename}'")
print(f"Total baris akhir: {len(df_clean):,}")

# Menampilkan cuplikan data bersih
df_clean.head()


✅ Proses Wrangling Selesai!
Data bersih berhasil disimpan ke: 'cleaned_data_v8.csv'
Total baris akhir: 111,480


,user_id,date,amount,type,category,subcategory,payment_method,description
0,USER_001,2021-01-01,5000000,income,Salary,gaji bulanan,debit,Pembayaran gaji bulanan
1,USER_001,2021-01-01,14500,expense,Others,donasi,debit,Pembayaran donasi
2,USER_001,2021-01-01,79000,expense,Food,fast food,e-wallet,Pembayaran fast food
3,USER_001,2021-01-01,50000,expense,Bills,pulsa,debit,Pembayaran pulsa
4,USER_001,2021-01-02,10000,expense,Food,kopi,cash,Pembayaran kopi
